In [1]:
import json
import copy
from collections import Counter, defaultdict, deque


def deepcopy_grid(g):
    return [row[:] for row in g]

def shape(grid):
    return len(grid), len(grid[0])

def same_grid(a, b):
    if len(a) != len(b):
        return False
    if len(a) == 0:
        return len(b) == 0
    if len(a[0]) != len(b[0]):
        return False
    return all(r1 == r2 for r1, r2 in zip(a, b))

def count_colors(grid):
    c = Counter()
    for row in grid:
        c.update(row)
    return c

def most_common_color(grid):
    return count_colors(grid).most_common(1)[0][0]

def non_bg_cells(grid, bg=None):
    if bg is None:
        bg = most_common_color(grid)
    cells = []
    for i, row in enumerate(grid):
        for j, v in enumerate(row):
            if v != bg:
                cells.append((i, j))
    return cells

def bbox_of_cells(cells):
    if not cells:
        return None
    rs = [r for r, c in cells]
    cs = [c for r, c in cells]
    return min(rs), max(rs), min(cs), max(cs)

def crop_bbox(grid, bbox):
    r1, r2, c1, c2 = bbox
    return [row[c1:c2+1] for row in grid[r1:r2+1]]

def crop_non_bg(grid, bg=None):
    if bg is None:
        bg = most_common_color(grid)
    cells = non_bg_cells(grid, bg)
    if not cells:
        return [[bg]]
    return crop_bbox(grid, bbox_of_cells(cells))

def rotate90(grid):
    h, w = shape(grid)
    return [[grid[h - 1 - r][c] for r in range(h)] for c in range(w)]

def rotate180(grid):
    return [row[::-1] for row in grid[::-1]]

def rotate270(grid):
    h, w = shape(grid)
    return [[grid[r][w - 1 - c] for r in range(h)] for c in range(w)][::-1]

def flip_h(grid):
    return [row[::-1] for row in grid]

def flip_v(grid):
    return grid[::-1]

def transpose(grid):
    h, w = shape(grid)
    return [[grid[r][c] for r in range(h)] for c in range(w)]

def replace_colors(grid, mapping):
    out = []
    for row in grid:
        out.append([mapping.get(v, v) for v in row])
    return out

def unique_colors(grid):
    s = set()
    for row in grid:
        for x in row:
            s.add(x)
    return s

def tile(grid, times_r, times_c):
    out = []
    for _ in range(times_r):
        for row in grid:
            out.append(row * times_c)
    return out

def scale2(grid):
    out = []
    for row in grid:
        expanded = []
        for v in row:
            expanded.extend([v, v])
        out.append(expanded[:])
        out.append(expanded[:])
    return out

def fill_bbox_with_color(grid, color, bg=None):
    if bg is None:
        bg = most_common_color(grid)
    cells = non_bg_cells(grid, bg)
    if not cells:
        return deepcopy_grid(grid)
    r1, r2, c1, c2 = bbox_of_cells(cells)
    out = deepcopy_grid(grid)
    for r in range(r1, r2 + 1):
        for c in range(c1, c2 + 1):
            out[r][c] = color
    return out

def get_connected_components(grid, bg=None):
    if bg is None:
        bg = most_common_color(grid)
    h, w = shape(grid)
    seen = [[False] * w for _ in range(h)]
    comps = []

    for r in range(h):
        for c in range(w):
            if seen[r][c] or grid[r][c] == bg:
                continue
            color = grid[r][c]
            q = deque([(r, c)])
            seen[r][c] = True
            cells = []
            while q:
                x, y = q.popleft()
                cells.append((x, y))
                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < h and 0 <= ny < w and not seen[nx][ny] and grid[nx][ny] == color:
                        seen[nx][ny] = True
                        q.append((nx, ny))
            comps.append({"color": color, "cells": cells})
    return comps

def largest_component_crop(grid, bg=None):
    comps = get_connected_components(grid, bg)
    if not comps:
        return crop_non_bg(grid, bg)
    comp = max(comps, key=lambda z: len(z["cells"]))
    return crop_bbox(grid, bbox_of_cells(comp["cells"]))

def smallest_component_crop(grid, bg=None):
    comps = get_connected_components(grid, bg)
    if not comps:
        return crop_non_bg(grid, bg)
    comp = min(comps, key=lambda z: len(z["cells"]))
    return crop_bbox(grid, bbox_of_cells(comp["cells"]))

def infer_background_from_pair(inp, out):
    cin = count_colors(inp)
    cout = count_colors(out)
    return cin.most_common(1)[0][0]



class Candidate:
    def __init__(self, name, fn, priority=0):
        self.name = name
        self.fn = fn
        self.priority = priority

    def __call__(self, grid):
        return self.fn(grid)

def build_basic_candidates(train_pairs):
    cands = []

    cands.append(Candidate("identity", lambda g: deepcopy_grid(g), 100))
    cands.append(Candidate("rot90", rotate90, 50))
    cands.append(Candidate("rot180", rotate180, 50))
    cands.append(Candidate("rot270", rotate270, 50))
    cands.append(Candidate("flip_h", flip_h, 50))
    cands.append(Candidate("flip_v", flip_v, 50))
    cands.append(Candidate("transpose", transpose, 40))
    cands.append(Candidate("crop_non_bg", crop_non_bg, 80))
    cands.append(Candidate("largest_component_crop", largest_component_crop, 70))
    cands.append(Candidate("smallest_component_crop", smallest_component_crop, 60))
    cands.append(Candidate("scale2", scale2, 20))

    color_map = infer_global_color_map(train_pairs)
    if color_map is not None:
        cands.append(Candidate(f"global_color_map_{color_map}", lambda g, m=color_map: replace_colors(g, m), 90))

    if color_map is not None:
        cands.append(Candidate(
            f"crop_non_bg_then_color_map_{color_map}",
            lambda g, m=color_map: replace_colors(crop_non_bg(g), m),
            75
        ))

    bbox_fill_color = infer_fill_bbox_color(train_pairs)
    if bbox_fill_color is not None:
        cands.append(Candidate(
            f"fill_bbox_with_{bbox_fill_color}",
            lambda g, col=bbox_fill_color: fill_bbox_with_color(g, col),
            30
        ))

    tiling = infer_tiling(train_pairs)
    if tiling is not None:
        tr, tc = tiling
        cands.append(Candidate(f"tile_{tr}x{tc}", lambda g, tr=tr, tc=tc: tile(g, tr, tc), 35))

    return cands

def infer_global_color_map(train_pairs):
    mapping = {}
    for pair in train_pairs:
        inp = pair["input"]
        out = pair["output"]
        if shape(inp) != shape(out):
            return None
        local = {}
        for r in range(len(inp)):
            for c in range(len(inp[0])):
                a = inp[r][c]
                b = out[r][c]
                if a in local and local[a] != b:
                    return None
                local[a] = b
        for k, v in local.items():
            if k in mapping and mapping[k] != v:
                return None
            mapping[k] = v
    return mapping if mapping else None

def infer_fill_bbox_color(train_pairs):
    inferred = None
    for pair in train_pairs:
        out_colors = count_colors(pair["output"])
        if len(out_colors) > 2:
            return None
        common = out_colors.most_common(1)[0][0]
        if inferred is None:
            inferred = common
        elif inferred != common:
            return None
    return inferred

def infer_tiling(train_pairs):
    ratios = []
    for pair in train_pairs:
        hi, wi = shape(pair["input"])
        ho, wo = shape(pair["output"])
        if hi == 0 or wi == 0:
            return None
        if ho % hi != 0 or wo % wi != 0:
            return None
        ratios.append((ho // hi, wo // wi))
    if not ratios:
        return None
    if len(set(ratios)) == 1:
        return ratios[0]
    return None


def candidate_fits_train(candidate, train_pairs):
    for pair in train_pairs:
        pred = safe_apply(candidate, pair["input"])
        if pred is None:
            return False
        if not same_grid(pred, pair["output"]):
            return False
    return True

def safe_apply(candidate, grid):
    try:
        out = candidate(grid)
        if not is_valid_grid(out):
            return None
        return out
    except Exception:
        return None

def is_valid_grid(grid):
    if not isinstance(grid, list) or len(grid) == 0:
        return False
    if not all(isinstance(row, list) and len(row) > 0 for row in grid):
        return False
    w = len(grid[0])
    for row in grid:
        if len(row) != w:
            return False
        for x in row:
            if not isinstance(x, int) or not (0 <= x <= 9):
                return False
    if len(grid) > 30 or w > 30:
        return False
    return True

def heuristic_score(candidate, train_pairs):
    score = candidate.priority

    for pair in train_pairs:
        pred = safe_apply(candidate, pair["input"])
        if pred is None:
            return -10**9

        ho, wo = shape(pair["output"])
        hp, wp = shape(pred)

        if (ho, wo) == (hp, wp):
            score += 10
        else:
            score -= abs(ho - hp) + abs(wo - wp)

        out_colors = set(unique_colors(pair["output"]))
        pred_colors = set(unique_colors(pred))
        score += 2 * len(out_colors & pred_colors) - len(out_colors ^ pred_colors)

    return score


def solve_task(task):
    train_pairs = task["train"]
    test_pairs = task["test"]

    candidates = build_basic_candidates(train_pairs)

    exact = []
    nonexact = []

    for cand in candidates:
        if candidate_fits_train(cand, train_pairs):
            exact.append(cand)
        else:
            nonexact.append(cand)

    exact = sorted(exact, key=lambda c: heuristic_score(c, train_pairs), reverse=True)
    nonexact = sorted(nonexact, key=lambda c: heuristic_score(c, train_pairs), reverse=True)

    chosen = []
    used_names = set()

    for cand in exact:
        if cand.name not in used_names:
            chosen.append(cand)
            used_names.add(cand.name)
        if len(chosen) == 2:
            break

    for cand in nonexact:
        if len(chosen) == 2:
            break
        if cand.name not in used_names:
            chosen.append(cand)
            used_names.add(cand.name)

    # fallback
    while len(chosen) < 2:
        fallback = Candidate("identity_fallback", lambda g: deepcopy_grid(g), 0)
        chosen.append(fallback)

    answers = []
    for test_item in test_pairs:
        inp = test_item["input"]

        pred1 = safe_apply(chosen[0], inp)
        pred2 = safe_apply(chosen[1], inp)

        if pred1 is None:
            pred1 = deepcopy_grid(inp)
        if pred2 is None:
            pred2 = deepcopy_grid(inp)

        answers.append({
            "attempt_1": pred1,
            "attempt_2": pred2
        })

    return answers


def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f)

def main():
    challenges_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"


    challenges = load_json(challenges_path)

    submission = {}
    for task_id, task in challenges.items():
        submission[task_id] = solve_task(task)

    save_json(submission, "submission.json")
    print("Saved submission.json")

if __name__ == "__main__":
    main()

Saved submission.json
